## Interval Analysis

In [21]:
# !pip install tensorboardX

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")
print(device)
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Simple NN. You can change this if you want. If you change it, mention the architectural details in your report.
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(28*28, 200)
        self.fc2 = nn.Linear(200,10)

    def forward(self, x):
        x = x.view((-1, 28*28))
        x = F.relu(self.fc(x))
        x = self.fc2(x)
        x = F.softmax(x, dim=-1) # added softmax for probabilities
        return x

    def forward_linear(self, lower, upper): # modified forward for interval analysis
        lower = lower.view(-1, 28*28)
        upper = upper.view(-1, 28*28)

        center = (lower + upper) / 2
        diff = (upper - lower) / 2
        lower, upper = linear_bounds(self.fc.weight, self.fc.bias, center, diff) 
        lower, upper = F.relu(lower), F.relu(upper) 
        return lower, upper

    def output_layer(self, lower, upper, labels): 
        W = self.fc2.weight
        b = self.fc2.bias
        W_y = W[labels]
        b_y = b[labels]

        j = torch.tensor(
            [[j for j in range(10) if j != y.item()] for y in labels], # exclude j = y
            device=labels.device)

        W_diff = W_y.unsqueeze(1) - W[j] # W_y - W_j
        b_diff = b_y.unsqueeze(1) - b[j] # b_y - b_j 
    
        center = (lower + upper) / 2
        diff = (upper - lower) / 2
        return linear_bounds(W_diff, b_diff, center, diff)

class Normalize(nn.Module):
    def forward(self, x):
        return (x - 0.1307)/0.3081

# Add the data normalization as a first "layer" to the network
# this allows us to search for adverserial examples to the real image, rather than
# to the normalized image
model = nn.Sequential(Normalize(), Net())

model = model.to(device)
model.train()


cuda


Sequential(
  (0): Normalize()
  (1): Net(
    (fc): Linear(in_features=784, out_features=200, bias=True)
    (fc2): Linear(in_features=200, out_features=10, bias=True)
  )
)

In [22]:
def train_model(model, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}')

def test_model(model):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Accuracy on images: {100 * correct / total}')

In [23]:
train_model(model, 15)
test_model(model)

Epoch 1/15, Loss: 2.012
Epoch 2/15, Loss: 1.729
Epoch 3/15, Loss: 1.627
Epoch 4/15, Loss: 1.596
Epoch 5/15, Loss: 1.582
Epoch 6/15, Loss: 1.572
Epoch 7/15, Loss: 1.566
Epoch 8/15, Loss: 1.561
Epoch 9/15, Loss: 1.557
Epoch 10/15, Loss: 1.553
Epoch 11/15, Loss: 1.550
Epoch 12/15, Loss: 1.547
Epoch 13/15, Loss: 1.545
Epoch 14/15, Loss: 1.542
Epoch 15/15, Loss: 1.540
Accuracy on images: 93.43


### Write the interval analysis for the simple model

In [24]:
## TODO: Write the interval analysis for the simple model
## you can use https://github.com/Zinoex/bound_propagation

def linear_bounds(weight, bias, center, diff): # based on ibp_forward_linear_jit
    center = center.unsqueeze(-2)
    diff = diff.unsqueeze(-2)
    weight = weight.transpose(-1, -2)

    w_mid = center.matmul(weight)
    if bias is not None:
        w_mid = w_mid + bias.unsqueeze(-2)
    
    w_diff = diff.matmul(weight.abs())

    lower = (w_mid - w_diff).squeeze(-2)
    upper = (w_mid + w_diff).squeeze(-2)

    return lower, upper


def interval_analysis(model, images, labels, epsilon):
    lower = torch.clamp(images - epsilon, 0, 1)
    upper = torch.clamp(images + epsilon, 0, 1)

    normalize, net = model
    lower = normalize(lower)
    upper = normalize(upper)

    lower, upper = net.forward_linear(lower, upper) 
    lower, upper = net.output_layer(lower, upper, labels) 

    return (lower >= 0).all(dim=1)


def test_interval(model): # modified test_model for interval analysis
    model.eval()

    eps = np.linspace(0.01, 0.1, 10)
    with torch.no_grad():
        for ep in eps:
            verified = 0
            total = 0
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                results = interval_analysis(model, images, labels, ep)
                total += labels.size(0)
                verified += results.sum().item()

            print(f'Epsilon: {ep:.2f}, Verified Accuracy: {100 * verified / total}%')

In [25]:
test_interval(model)

Epsilon: 0.01, Verified Accuracy: 70.55%
Epsilon: 0.02, Verified Accuracy: 23.68%
Epsilon: 0.03, Verified Accuracy: 4.56%
Epsilon: 0.04, Verified Accuracy: 0.58%
Epsilon: 0.05, Verified Accuracy: 0.02%
Epsilon: 0.06, Verified Accuracy: 0.0%
Epsilon: 0.07, Verified Accuracy: 0.0%
Epsilon: 0.08, Verified Accuracy: 0.0%
Epsilon: 0.09, Verified Accuracy: 0.0%
Epsilon: 0.10, Verified Accuracy: 0.0%
